# HDMI 增强面板（640×480）

**接线**：HDMI 采集卡插在板子的 **HDMI OUT**，USB 接电脑；电脑上打开 OBS 看采集源。
不需要 HDMI 输入——这一版由 PS 送帧，内核在真实像素通路上处理。

## 用法

1. 从上到下依次运行单元格；
2. 最后一个单元会出现一排按钮，点哪个就切到哪个预设；
3. 每次切换后输出链路会自动重建（约 0.2 秒），之后画面稳定。

> 说明：本板 HDMI 输出只能驱动 640×480（800×600/720p/1080p 下 VDMA 一帧都不消费，
> 原厂位流同样如此）。切换预设时需要复位内核让 HLS 重新采样配置，
> 所以每次点按钮会有一次约 0.2 秒的重连，这是已知代价。


In [ ]:
import time
import numpy as np
from pynq import Overlay
from pynq.lib.video import VideoMode

WIDTH, HEIGHT = 640, 480

ol = Overlay("enhance_hdmi.bit")
video = ol.video
out = video.hdmi_out

hpd = None
for name in ol.ip_dict:
    if "gpio_hdmiin" in name.lower():
        hpd = getattr(ol, name)
if hpd is not None:
    hpd.mmio.write(0x00, 0x1)      # 让 Windows 重新枚举 2 号屏

cfg = None
lat = None
for name in ol.ip_dict:
    if "gpio_enh_cfg" in name.lower():
        cfg = getattr(ol, name)
    if "gpio_enh_lat" in name.lower():
        lat = getattr(ol, name)
print("cfg:", cfg is not None, " latency:", lat is not None)


## 预设与打包

In [ ]:
# 59 位配置：word0 = clarity|knee|gain_lo|gain_hi，word1 = thresh|binarize|invert|mono|split_x
PRESETS = {
    "pass":     dict(clarity=16, knee=255, gain_lo=16, gain_hi=16),
    "clarity":  dict(clarity=28, knee=255, gain_lo=16, gain_hi=16),
    "reading":  dict(clarity=24, knee=48, gain_lo=88, gain_hi=14),
    "document": dict(clarity=20, knee=192, gain_lo=64, gain_hi=12),
    "night":    dict(clarity=30, knee=36, gain_lo=120, gain_hi=10),
    "invert":   dict(clarity=24, knee=48, gain_lo=88, gain_hi=14, invert=1, mono=1),
    "binarize": dict(clarity=16, knee=64, gain_lo=112, gain_hi=8,
                      thresh=112, binarize=1, mono=1),
    "split":    dict(clarity=28, knee=90, gain_lo=40, gain_hi=12, split_x=320),
}


def pack(p):
    lo = ((p.get("clarity", 16) & 0xFF)
          | ((p.get("knee", 255) & 0xFF) << 8)
          | ((p.get("gain_lo", 16) & 0xFF) << 16)
          | ((p.get("gain_hi", 16) & 0xFF) << 24))
    hi = ((p.get("thresh", 0) & 0xFF)
          | ((p.get("binarize", 0) & 1) << 8)
          | ((p.get("invert", 0) & 1) << 9)
          | ((p.get("mono", 0) & 1) << 10)
          | ((p.get("split_x", 0) & 0xFFFF) << 11))
    return lo, hi

print(len(PRESETS), "presets")


## 测试图与输出链路

In [ ]:
def test_image():
    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    ramp = np.linspace(20, 235, HEIGHT, dtype=np.uint8).reshape(HEIGHT, 1)
    img[:, :, 0] = ramp
    img[:, :, 1] = ramp
    img[:, :, 2] = ramp
    top = HEIGHT // 2 - 60
    img[top:top + 120, :, :] = 128
    for row in range(top + 10, top + 110, 22):
        left = 40
        while left < WIDTH - 60:
            run = int(30 + (left * 7) % 90)
            img[row:row + 10, left:min(left + run, WIDTH - 40), :] = 25
            left += run + 18
    img[10:70, 10:130, 2] = 220
    img[10:70, 140:260, 1] = 220
    img[10:70, 270:390, 0] = 220
    return img


IMAGE = test_image()


def apply_preset(name):
    """写配置 → 复位内核 → 重建输出链路 → 重新填满 4 块帧缓冲。"""
    lo, hi = pack(PRESETS[name])
    cfg.mmio.write(0x00, lo)
    cfg.mmio.write(0x08, hi | (1 << 31))   # 请求内核复位
    time.sleep(0.002)
    cfg.mmio.write(0x08, hi)               # 释放，prologue 采样新值
    time.sleep(0.002)
    out.stop()
    out.configure(VideoMode(WIDTH, HEIGHT, 24, 60))
    out.start()
    for _ in range(4):
        f = out.newframe()
        f[:] = IMAGE
        out.writeframe(f)
    return read_latency()


def read_latency():
    a = lat.mmio.read(0x00)
    b = lat.mmio.read(0x08)
    if a == 0 and b == 0:
        return None
    return (b - a) & 0xFFFFFFFF

print("ready")


## 一键切换（点按钮）

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

status = widgets.Output()


def make_button(name):
    button = widgets.Button(description=name, layout=widgets.Layout(width="110px"))

    def on_click(_):
        n = apply_preset(name)
        with status:
            clear_output(wait=True)
            shown = "未测到" if n is None else f"{n} 个像素时钟"
            print(f"预设 {name} 已生效，流水线延迟 {shown}")

    button.on_click(on_click)
    return button


buttons = [make_button(n) for n in PRESETS]
display(widgets.VBox([widgets.HBox(buttons[:4]),
                      widgets.HBox(buttons[4:]), status]))


## 连续演示（自动循环，每档 15 秒）

In [ ]:
import time

ROUNDS = 1
HOLD = 15

order = ["pass", "clarity", "reading", "document",
         "night", "invert", "binarize", "split"]

for _ in range(ROUNDS):
    for name in order:
        n = apply_preset(name)
        shown = "未测到" if n is None else f"{n} 拍"
        print(f"[{time.strftime('%H:%M:%S')}] {name:<9} 延迟 {shown}")
        time.sleep(HOLD)
print("演示结束")
